# COSC2753 Assignment 2 — Fashion Intelligence System
## Task 4 — Image-Only Test-Set Prediction


## Part I — Configuration, paths and the preprocessing artifacts

Only two preprocessing artifacts are needed now:

| Artifact | File | Used for |
|---|---|---|
| Image size and normalisation constants | `pipeline_config.json` | rebuilding `eval_transform` |
| Fitted `LabelEncoder` per target | `label_encoders.pkl` | decoding the `season` model's output |

`train_full.csv`, `val_full.csv` and the per-task holdout subsets are no longer loaded — they
exist to train and validate, and this notebook does neither. The transform is rebuilt from the
recorded `normalization_mean` / `normalization_std` so it is numerically identical to the one
the models were trained under, not merely similar.

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.preprocessing import LabelEncoder   # needed to unpickle label_encoders.pkl

DATA_DIR = Path("../data/raw/FashionDataset")
OUT_DIR = Path("../data/processed")

TEST_PRED_CSV = DATA_DIR / "test" / "styles_prediction.csv"
IMAGES_TEST_DIR = DATA_DIR / "test" / "images_test"

REQUIRED = {
    "config":   OUT_DIR / "pipeline_config.json",
    "encoders": OUT_DIR / "label_encoders.pkl",
}
for name, path in REQUIRED.items():
    print(f"[{'OK' if path.exists() else 'MISSING':>7}] {name:12s} {path}")
missing = {k: v for k, v in REQUIRED.items() if not v.exists()}
if missing:
    raise FileNotFoundError(
        "Run COSC2753_A2_Preprocessing.ipynb first -- it writes the files above. "
        f"Missing: {list(missing)}")

for p in [TEST_PRED_CSV, IMAGES_TEST_DIR]:
    print(f"[{'OK' if p.exists() else 'MISSING':>7}] {p}")

with open(REQUIRED["config"]) as f:
    config = json.load(f)

RANDOM_STATE = config["random_state"]
IMG_WIDTH = config["image"]["width"]
IMG_HEIGHT = config["image"]["height"]
mean = torch.tensor(config["image"]["normalization_mean"])
std = torch.tensor(config["image"]["normalization_std"])

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

with open(REQUIRED["encoders"], "rb") as f:
    encoders = pickle.load(f)

print(f"\nSeed {RANDOM_STATE} | images {IMG_HEIGHT}x{IMG_WIDTH} (HxW)")
print(f"Normalisation mean {[round(v, 4) for v in mean.tolist()]}, "
      f"std {[round(v, 4) for v in std.tolist()]}")
print(f"Label encoders available: {list(encoders)}")

[     OK] config       ..\data\processed\pipeline_config.json
[     OK] encoders     ..\data\processed\label_encoders.pkl
[     OK] ..\data\raw\FashionDataset\test\styles_prediction.csv
[     OK] ..\data\raw\FashionDataset\test\images_test

Seed 42 | images 80x60 (HxW)
Normalisation mean [0.8476, 0.8307, 0.8243], std [0.274, 0.285, 0.2889]
Label encoders available: ['articleType', 'season', 'gender', 'usage']


### The evaluation transform

Only `eval_transform` is rebuilt. The augmented `train_transform` (flips, rotation, colour
jitter) exists to regularise training and must never touch test images — augmenting at
inference would change the prediction for no reason. `FashionImageDataset`,
`get_weighted_sampler` and the class-weight tensors are all training-time machinery and are
dropped with it.

In [2]:
def to_rgb(img):
    """Converts the grayscale ('L' mode) files to 3-channel. A module-level function,
    not a lambda, so the transform stays picklable for DataLoader(num_workers>0)."""
    return img.convert("RGB")


eval_transform = T.Compose([
    to_rgb,
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.ToTensor(),
    T.Normalize(mean=mean.tolist(), std=std.tolist()),
])

test_df = pd.read_csv(TEST_PRED_CSV)
test_df['id'] = test_df['id'].astype(str).str.strip()
TEST_TEMPLATE_COLUMNS = list(test_df.columns)
TEST_IDS = test_df['id'].tolist()
print(f"Test template: {test_df.shape[0]} rows, columns {TEST_TEMPLATE_COLUMNS}")

_probe = eval_transform(Image.open(IMAGES_TEST_DIR / f"{TEST_IDS[0]}.jpg"))
assert _probe.shape == (3, IMG_HEIGHT, IMG_WIDTH), f"unexpected tensor shape {tuple(_probe.shape)}"
print(f"Transform check: test image {TEST_IDS[0]} -> tensor {tuple(_probe.shape)}, "
      f"range [{_probe.min():.2f}, {_probe.max():.2f}]")

Test template: 5829 rows, columns ['id', 'gender', 'articleType', 'season', 'usage']
Transform check: test image 52003 -> tensor (3, 80, 60), range [-3.09, 0.61]


## Part II — The four image-only models

### 1. Artifact paths

This notebook trains nothing; it only loads checkpoints written by Tasks 1-3. Four
checkpoints and their label encoders, one per target column:

| Column | Checkpoint | From |
|---|---|---|
| `articleType` | `best_imageonly_articletype.pt` | Task 1 |
| `season` | `improved_small_cnn_image_only.pt` | Task 2 |
| `gender` | `gender_imageonly_best.pt` | Task 3 |
| `usage` | `usage_imageonly_best.pt` | Task 3 |

The multi-input checkpoints, the metadata preprocessors (`metadata_preprocessor_task1.joblib`,
`meta_preprocessor_task2.joblib`, `ohe_metadata.joblib`) and the whole BaseColour notebook are
no longer required — nothing here consumes metadata, so none of them is listed.

In [3]:
import joblib
from torch import nn
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, NUM_WORKERS = 128, 0
print("Device:", DEVICE)

TASK1_DIR = Path("../outputs/task1_models")
TASK2_DIR = Path("../outputs/task2_models")
TASK3_DIR = Path("../outputs/task3_models")
OUTPUT_DIR = Path("../outputs/image_only")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fail early and specifically rather than deep inside a load call.
REQUIRED_ARTIFACTS = {
    "Task 1 image-only articleType": TASK1_DIR / "best_imageonly_articletype.pt",
    "Task 1 articleType encoder":    TASK1_DIR / "articletype_encoder_task1.joblib",
    "Task 2 image-only season":      TASK2_DIR / "improved_small_cnn_image_only.pt",
    "Task 3 gender image-only":      TASK3_DIR / "image_only/gender_imageonly_best.pt",
    "Task 3 usage image-only":       TASK3_DIR / "image_only/usage_imageonly_best.pt",
    "Task 3 gender encoder":         TASK3_DIR / "gender_encoder.joblib",
    "Task 3 usage encoder":          TASK3_DIR / "usage_encoder.joblib",
}

for name, path in REQUIRED_ARTIFACTS.items():
    print(f"[{'OK' if path.exists() else 'MISSING':>7}] {name:32s} {path}")

missing = {k: v for k, v in REQUIRED_ARTIFACTS.items() if not v.exists()}
if missing:
    raise FileNotFoundError(
        "Run Tasks 1-3 to completion first -- they produce the artifacts above. "
        f"Missing: {list(missing)}")
print("\nAll required artifacts present.")

Device: cpu
[     OK] Task 1 image-only articleType    ..\outputs\task1_models\best_imageonly_articletype.pt
[     OK] Task 1 articleType encoder       ..\outputs\task1_models\articletype_encoder_task1.joblib
[     OK] Task 2 image-only season         ..\outputs\task2_models\improved_small_cnn_image_only.pt
[     OK] Task 3 gender image-only         ..\outputs\task3_models\image_only\gender_imageonly_best.pt
[     OK] Task 3 usage image-only          ..\outputs\task3_models\image_only\usage_imageonly_best.pt
[     OK] Task 3 gender encoder            ..\outputs\task3_models\gender_encoder.joblib
[     OK] Task 3 usage encoder             ..\outputs\task3_models\usage_encoder.joblib

All required artifacts present.


### 2. Architecture definitions

A checkpoint is only weights; the class it was saved from has to exist here for
`load_state_dict` to have anything to load into. These definitions are copied verbatim from
the task notebooks — if one drifts, the `strict=True` load below fails loudly instead of
silently loading a subset of the weights.

Only the image encoders and the three image-only wrappers are kept. `MetadataEncoder`,
`MultiInputNetT1` and `MultiInputNetT23` are gone along with the models that used them.

In [4]:
import torch.nn.functional as F
from torchvision.models import resnet18

# ── From Tasks 1 and 2: SE-residual building blocks ──────────────────────────
class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        s = torch.sigmoid(self.fc2(F.silu(self.fc1(s))))
        return x * s[:, :, None, None]


class SEResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.se = SqueezeExcite(out_ch)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.shortcut = (nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
                         if (stride != 1 or in_ch != out_ch) else nn.Identity())

    def forward(self, x):
        out = F.silu(self.bn1(x))
        shortcut = self.shortcut(out if not isinstance(self.shortcut, nn.Identity) else x)
        out = self.conv1(out)
        out = self.conv2(F.silu(self.bn2(out)))
        out = self.se(self.drop(out))
        return out + shortcut


class SEResidualCNN(nn.Module):
    """Task 1's name for the encoder."""
    def __init__(self, out_dim=128, widths=(32, 64, 128, 256), drop=0.1):
        super().__init__()
        self.stem = nn.Conv2d(3, widths[0], 3, padding=1, bias=False)
        stages, in_ch = [], widths[0]
        for stage_idx, width in enumerate(widths):
            stride = 1 if stage_idx == 0 else 2
            stages.append(SEResidualBlock(in_ch, width, stride=stride, drop=drop))
            stages.append(SEResidualBlock(width, width, stride=1, drop=drop))
            in_ch = width
        self.stages = nn.Sequential(*stages)
        self.norm = nn.BatchNorm2d(in_ch)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_ch, out_dim))

    def forward(self, x):
        x = self.stages(self.stem(x))
        x = self.pool(F.silu(self.norm(x))).flatten(1)
        return self.proj(x)


SEResidualEncoder = SEResidualCNN   # Task 2's name for the identical architecture


# ── From Task 1 ──────────────────────────────────────────────────────────────
class BasicCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.proj(self.pool(self.features(x)).flatten(1))

class ResNetStyleCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = resnet18(weights=None)          # from scratch, as in Task 1
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.out_dim = out_dim
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class Classifier(nn.Module):
    """Task 1's image-only wrapper."""
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.out_dim, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


# ── From Tasks 2 and 3 ───────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + residual)


class ImprovedSmallImageEncoder(nn.Module):
    def __init__(self, out_dim=128, dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.stage1 = ConvBlock(3, 32);   self.pool1 = nn.MaxPool2d(2)
        self.stage2 = ConvBlock(32, 64);  self.pool2 = nn.MaxPool2d(2)
        self.stage3 = ConvBlock(64, 128); self.pool3 = nn.MaxPool2d(2)
        self.stage4 = ConvBlock(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, out_dim),
                                  nn.BatchNorm1d(out_dim), nn.SiLU())

    def forward(self, x):
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        return self.proj(self.global_pool(self.stage4(x)).flatten(1))


class SmallImageEncoder(nn.Module):
    """Task 2's architecture -- note it differs from Task 3's class of the same name."""
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.out_dim = out_dim
        self.proj = nn.Linear(128, out_dim)

    def forward(self, x):
        return self.proj(self.features(x).flatten(1))


class ResNet18Encoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.out_dim = out_dim
        self.backbone = backbone
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class EfficientNetEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import efficientnet_b0
        backbone = efficientnet_b0(weights=None)
        self.out_dim = out_dim
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.proj = nn.Linear(1280, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class ImageOnlyClassifier(nn.Module):
    """Tasks 2 and 3's image-only wrapper."""
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.out_dim, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation block: pools global context per channel, then learns a
    per-channel gate so the network can emphasise the most informative channels."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced), nn.ReLU(inplace=True),
            nn.Linear(reduced, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        gate = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * gate


class SEConvBlock(nn.Module):
    """Residual ConvBlock with a Squeeze-and-Excitation gate before the skip connection."""
    def __init__(self, in_ch, out_ch, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.act   = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.se    = SEBlock(out_ch, reduction=reduction)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Sequential()
        )

    def forward(self, x):
        out = self.bn2(self.conv2(self.act(self.bn1(self.conv1(x)))))
        out = self.se(out)
        return self.act(out + self.shortcut(x))


class SEResidualImageEncoder(nn.Module):
    """Same stage layout as ImprovedSmallImageEncoder, every block an SEConvBlock."""
    def __init__(self, out_dim=128, dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.stage1 = SEConvBlock(3, 32);   self.pool1 = nn.MaxPool2d(2)
        self.stage2 = SEConvBlock(32, 64);  self.pool2 = nn.MaxPool2d(2)
        self.stage3 = SEConvBlock(64, 128); self.pool3 = nn.MaxPool2d(2)
        self.stage4 = SEConvBlock(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(256, out_dim),
            nn.BatchNorm1d(out_dim), nn.SiLU()
        )

    def forward(self, x):
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        return self.proj(self.global_pool(self.stage4(x)).flatten(1))


class ScratchResNet18Encoder(nn.Module):
    """ResNet18 with randomly-initialised weights -- a legitimate from-scratch candidate."""
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.out_dim  = out_dim
        self.proj     = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class Task3SmallImageEncoder(nn.Module):
    """Task 3's plain small CNN -- named separately from SmallImageEncoder above, which is
    Task 2's DIFFERENT architecture that happens to share the same class name there."""
    def __init__(self, out_dim=128, dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(256, out_dim),
            nn.BatchNorm1d(out_dim), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.features(x)
        return self.proj(self.global_pool(x).flatten(1))


class ImageOnlyNet(nn.Module):
    """Task 3's image-only wrapper -- an MLP head rather than a bare Linear."""
    def __init__(self, n_classes, image_encoder, dropout=0.3):
        super().__init__()
        self.image = image_encoder
        self.head = nn.Sequential(
            nn.Linear(image_encoder.out_dim, 256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, n_classes)
        )

    def forward(self, image):
        return self.head(self.image(image))


# Task 1 / Task 2 write `encoder_class`; Task 3 writes `image_encoder`. Both are read off
# the checkpoint rather than hardcoded, so whichever encoder won stays correct.
ENCODER_REGISTRY = {
    "SEResidualCNN": SEResidualCNN, "SEResidualEncoder": SEResidualEncoder,
    "BasicCNN": BasicCNN, "ResNetStyleCNN": ResNetStyleCNN,
    "ImprovedSmallImageEncoder": ImprovedSmallImageEncoder,
    "SmallImageEncoder": SmallImageEncoder, "ResNet18Encoder": ResNet18Encoder,
    "EfficientNetEncoder": EfficientNetEncoder,
}

IMG_ENCODER_REGISTRY = {
    "SmallImageEncoder": Task3SmallImageEncoder,             # Task 3's, not Task 2's
    "ImprovedSmallImageEncoder": ImprovedSmallImageEncoder,
    "SEResidualImageEncoder": SEResidualImageEncoder,
    "ScratchResNet18Encoder": ScratchResNet18Encoder,
}

print(f"{len(ENCODER_REGISTRY)} Task 1/2 encoders and {len(IMG_ENCODER_REGISTRY)} "
      "Task 3 encoders registered.")

8 Task 1/2 encoders and 4 Task 3 encoders registered.


### 3. Dataset, loader and the inference helper

One dataset class, images only. `shuffle=False` everywhere, so position *i* of a prediction
array corresponds to position *i* of the id list — that is the single assumption the whole
submission's alignment rests on, and it is checked directly in Section 5.

`predict_all_image_only` runs all four models in a **single pass** over the loader. Decoding
each JPEG once instead of four times is the bulk of the wall-clock cost on CPU; the models
themselves are small.

In [5]:
class InferenceImageDataset(Dataset):
    """Images only, no labels. Returns (image_tensor, id) in the order given."""
    def __init__(self, ids, images_dir, transform):
        self.ids = list(ids)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]
        with Image.open(self.images_dir / f"{img_id}.jpg") as im:
            return self.transform(im), img_id


def make_inference_loader(ids, images_dir):
    return DataLoader(InferenceImageDataset(ids, images_dir, eval_transform),
                      batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


@torch.no_grad()
def predict_all_image_only(models, loader):
    """Runs every model in `models` over `loader` in one pass.

    `models` maps name -> (model, encoder). Returns name -> array of predicted class
    INDICES in loader order.
    """
    for model, _ in models.values():
        model.eval()
    out = {name: [] for name in models}
    for imgs, _ in tqdm(loader, desc="image-only inference"):
        imgs = imgs.to(DEVICE)
        for name, (model, _) in models.items():
            out[name].append(model(imgs).argmax(1).cpu())
    return {name: torch.cat(parts).numpy() for name, parts in out.items()}


def extract_state(ck):
    """The weights, whether they were saved bare or wrapped in a metadata dict.

    The task notebooks are not consistent: some saved `{"model_state_dict": ..., "n_classes": ...}`,
    some saved the bare state dict. Both are handled here so nothing downstream has to care.
    """
    if isinstance(ck, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in ck and isinstance(ck[key], dict):
                return ck[key]
        if all(torch.is_tensor(v) for v in ck.values()):
            return ck
        return {k: v for k, v in ck.items() if torch.is_tensor(v)}
    return ck


def load_state(model, path_or_ckpt, label):
    """strict=True load with a clear failure message naming the culprit."""
    ckpt = torch.load(path_or_ckpt, map_location=DEVICE) if not isinstance(path_or_ckpt, dict) else path_or_ckpt
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as e:
        raise RuntimeError(
            f"{label}: saved weights do not match the architecture declared in Section 2. "
            f"This almost always means the encoder was edited in the task notebook after "
            f"the checkpoint was written -- retrain or sync the class definition.\n\n{e}")
    return model.to(DEVICE).eval()

### 4. What is actually inside each checkpoint

The task notebooks did not all save the same metadata alongside their weights — Task 1 writes
`encoder_class`, and Task 3's image-only checkpoints turn out not to record the encoder name at
all. Rather than hardcode an assumption that breaks the moment a checkpoint is re-saved, this
cell prints what each file really contains, and the loader below reads the architecture off the
**weights** when the metadata is absent.

The weights are unambiguous on their own: each candidate encoder has a distinct parameter
layout (`backbone.*` for the ResNet, `stage*.se.*` for the SE variant, `stage*.conv1.*` for the
improved block, `features.*` for the plain CNN), and `out_dim` / `n_classes` are just the shapes
of the projection and the final linear layer.

In [6]:
def describe_checkpoint(path, label):
    """Print the non-tensor metadata and the state-dict key structure of a checkpoint."""
    ck = torch.load(path, map_location="cpu")
    print(f"\n--- {label}  ({Path(path).name}) ---")
    if isinstance(ck, dict) and not all(torch.is_tensor(v) for v in ck.values()):
        meta = {k: v for k, v in ck.items() if not torch.is_tensor(v) and k != "model_state_dict"}
        print("  top-level keys :", list(ck))
        print("  metadata       :", {k: v for k, v in meta.items() if not isinstance(v, dict)})
    else:
        print("  bare state dict (no metadata saved)")
    state = extract_state(ck)
    prefixes = sorted({k.split('.')[0] for k in state})
    print("  state prefixes :", prefixes)
    print("  first 6 keys   :", list(state)[:6])
    print("  last 3 keys    :", list(state)[-3:])
    return ck


for label in ["Task 1 image-only articleType", "Task 2 image-only season",
              "Task 3 gender image-only", "Task 3 usage image-only"]:
    describe_checkpoint(REQUIRED_ARTIFACTS[label], label)


--- Task 1 image-only articleType  (best_imageonly_articletype.pt) ---
  top-level keys : ['model_state_dict', 'model_name', 'encoder_class', 'n_classes', 'out_dim', 'val_macro_f1', 'pretrained']
  metadata       : {'model_name': 'SE-Residual CNN', 'encoder_class': 'SEResidualCNN', 'n_classes': 95, 'out_dim': 128, 'val_macro_f1': 0.7012130032874487, 'pretrained': False}
  state prefixes : ['encoder', 'head']
  first 6 keys   : ['encoder.stem.weight', 'encoder.stages.0.bn1.weight', 'encoder.stages.0.bn1.bias', 'encoder.stages.0.bn1.running_mean', 'encoder.stages.0.bn1.running_var', 'encoder.stages.0.bn1.num_batches_tracked']
  last 3 keys    : ['encoder.proj.1.bias', 'head.weight', 'head.bias']

--- Task 2 image-only season  (improved_small_cnn_image_only.pt) ---
  top-level keys : ['model_state_dict', 'n_classes', 'out_dim', 'val_f1']
  metadata       : {'n_classes': 4, 'out_dim': 128, 'val_f1': 0.7375844608235447}
  state prefixes : ['encoder', 'head']
  first 6 keys   : ['encoder.st

### 5. Loading the four models

Architecture, `out_dim` and `n_classes` come from the checkpoint's metadata when it recorded
them and are inferred from the weight shapes when it did not. Either way the load is
`strict=True`, so a wrong guess fails immediately with a shape mismatch rather than quietly
loading a partial model.

Each load is followed by a forward pass on real test images and a logit-width check. A model
that loads cleanly but emits the wrong number of logits would otherwise only reveal itself much
later, as a confusing indexing error inside `inverse_transform` — or not at all, if the widths
happen to be compatible but the class order is not.

In [7]:
# ── Reading the architecture off the weights ─────────────────────────────────
# Each candidate encoder leaves a distinct fingerprint in the parameter names.
def infer_encoder_class(state, prefix, registry):
    """Identify the encoder from its parameter layout when no name was saved.

    Each candidate leaves a distinct fingerprint. A fingerprint can map to more than one
    registered class name (the two registries use different names for similar layouts), so
    the first candidate present in `registry` wins -- and if that guess is wrong, the
    strict=True load below fails loudly with a shape mismatch rather than half-loading.
    """
    keys = [k[len(prefix):] for k in state if k.startswith(prefix)]
    if not keys:
        raise KeyError(f"no parameters under prefix '{prefix}'; keys look like {list(state)[:5]}")
    if any(k.startswith("backbone.") for k in keys):
        candidates = ("ScratchResNet18Encoder", "ResNetStyleCNN", "ResNet18Encoder")
    elif any(".se." in k or ".se1." in k for k in keys):
        candidates = ("SEResidualImageEncoder", "SEResidualCNN", "SEResidualEncoder")
    elif any(k.startswith("stage1.conv1") for k in keys):
        candidates = ("ImprovedSmallImageEncoder",)
    elif any(k.startswith("features.") for k in keys):
        candidates = ("SmallImageEncoder", "BasicCNN")
    else:
        raise KeyError(f"unrecognised encoder layout; sample keys {keys[:8]}")
    name = next((c for c in candidates if c in registry), None)
    if name is None:
        raise KeyError(f"layout matches {candidates}, none of which is in {list(registry)}")
    return name, registry[name]


def infer_dims(state, encoder_prefix, head_keys):
    """out_dim from the encoder's projection, n_classes from the last linear in the head."""
    proj = [k for k in state
            if k.startswith(encoder_prefix) and k.endswith("weight") and state[k].ndim == 2]
    out_dim = state[proj[-1]].shape[0] if proj else None
    head_linears = [k for k in head_keys if k.endswith("weight") and state[k].ndim == 2]
    n_classes = state[head_linears[-1]].shape[0]
    if out_dim is None:
        out_dim = state[head_linears[0]].shape[1]
    return out_dim, n_classes


def build_task3_model(ck, label):
    """Task 3's image-only net. Uses saved metadata where present, weights otherwise."""
    state = extract_state(ck)
    meta = ck if isinstance(ck, dict) else {}
    name = next((meta[k] for k in ("image_encoder", "encoder_class", "encoder", "architecture",
                                   "arch", "model_name") if isinstance(meta.get(k), str)), None)
    if name and name in IMG_ENCODER_REGISTRY:
        enc_cls = IMG_ENCODER_REGISTRY[name]
    else:
        name, enc_cls = infer_encoder_class(state, "image.", IMG_ENCODER_REGISTRY)
    head_keys = [k for k in state if k.startswith("head.")]
    out_dim, n_classes = infer_dims(state, "image.", head_keys)
    out_dim = meta.get("out_dim", out_dim)
    n_classes = meta.get("n_classes", n_classes)
    source = "checkpoint metadata" if meta.get("image_encoder") else "inferred from weights"
    print(f"       {label}: {name}, out_dim={out_dim}, n_classes={n_classes}  ({source})")
    return ImageOnlyNet(n_classes, enc_cls(out_dim=out_dim))


# ── Load and verify ──────────────────────────────────────────────────────────
image_models = {}
probe_loader = make_inference_loader(TEST_IDS[:BATCH_SIZE], IMAGES_TEST_DIR)
probe_imgs, probe_ids = next(iter(probe_loader))


def register_model(name, model, encoder, n_expected):
    """Load-and-verify: run a real batch through and confirm the logit width."""
    with torch.no_grad():
        logits = model(probe_imgs.to(DEVICE))
    assert logits.shape[1] == n_expected, \
        f"{name}: model outputs {logits.shape[1]} classes, encoder has {n_expected}"
    image_models[name] = (model, encoder)
    img_encoder = getattr(model, "encoder", None) or getattr(model, "image", None)
    print(f"  [OK] {name:12s} {logits.shape[1]:3d} classes  ({type(img_encoder).__name__})")


print("Image-only models:")

# --- articleType (Task 1) ---
ck = torch.load(REQUIRED_ARTIFACTS["Task 1 image-only articleType"], map_location=DEVICE)
enc_at = joblib.load(REQUIRED_ARTIFACTS["Task 1 articleType encoder"])
state = extract_state(ck)
if isinstance(ck, dict) and "encoder_class" in ck:
    enc_name, enc_cls = ck["encoder_class"], ENCODER_REGISTRY[ck["encoder_class"]]
else:
    enc_name, enc_cls = infer_encoder_class(state, "encoder.", ENCODER_REGISTRY)
out_dim, n_classes = infer_dims(state, "encoder.", [k for k in state if k.startswith("head.")])
out_dim = ck.get("out_dim", out_dim) if isinstance(ck, dict) else out_dim
n_classes = ck.get("n_classes", n_classes) if isinstance(ck, dict) else n_classes
print(f"       articleType: {enc_name}, out_dim={out_dim}, n_classes={n_classes}")
m = Classifier(enc_cls(out_dim=out_dim), n_classes)
register_model("articleType", load_state(m, state, "Task 1 articleType"),
               enc_at, len(enc_at.classes_))

# --- season (Task 2) ---
season_ckpt_path = REQUIRED_ARTIFACTS["Task 2 image-only season"]
ck = torch.load(season_ckpt_path, map_location=DEVICE)
enc_se = encoders['season']
state = extract_state(ck)
if isinstance(ck, dict) and isinstance(ck.get("encoder_class"), str):
    enc_name, enc_cls = ck["encoder_class"], ENCODER_REGISTRY[ck["encoder_class"]]
elif "seresidual" in season_ckpt_path.name.lower():
    enc_name, enc_cls = "SEResidualEncoder", SEResidualEncoder
else:
    enc_name, enc_cls = "ImprovedSmallImageEncoder", ImprovedSmallImageEncoder
out_dim, n_classes = infer_dims(state, "encoder.", [k for k in state if k.startswith("head.")])
out_dim = ck.get("out_dim", out_dim) if isinstance(ck, dict) else out_dim
n_classes = ck.get("n_classes", n_classes) if isinstance(ck, dict) else n_classes
print(f"       season: {enc_name}, out_dim={out_dim}, n_classes={n_classes}")
m = ImageOnlyClassifier(enc_cls(out_dim=out_dim), n_classes)
register_model("season", load_state(m, state, "Task 2 season"), enc_se, len(enc_se.classes_))

# --- gender and usage (Task 3) ---
for attr, ck_key, enc_key in [("gender", "Task 3 gender image-only", "Task 3 gender encoder"),
                              ("usage",  "Task 3 usage image-only",  "Task 3 usage encoder")]:
    ck = torch.load(REQUIRED_ARTIFACTS[ck_key], map_location=DEVICE)
    enc = joblib.load(REQUIRED_ARTIFACTS[enc_key])
    m = build_task3_model(ck, attr)
    register_model(attr, load_state(m, extract_state(ck), f"Task 3 {attr}"),
                   enc, len(enc.classes_))

TARGETS = ['articleType', 'season', 'gender', 'usage']
assert set(image_models) == set(TARGETS), f"expected {TARGETS}, got {list(image_models)}"
print(f"\nLoaded {len(image_models)} image-only models, one per target column.")

Image-only models:
       articleType: SEResidualCNN, out_dim=128, n_classes=95


d:\RMIT\ml_mas\COSC2753-Assignment-2\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  [OK] articleType   95 classes  (SEResidualCNN)
       season: ImprovedSmallImageEncoder, out_dim=128, n_classes=4
  [OK] season         4 classes  (ImprovedSmallImageEncoder)
       gender: ImprovedSmallImageEncoder, out_dim=128, n_classes=5  (inferred from weights)
  [OK] gender         5 classes  (ImprovedSmallImageEncoder)
       usage: SmallImageEncoder, out_dim=128, n_classes=7  (inferred from weights)
  [OK] usage          7 classes  (Task3SmallImageEncoder)

Loaded 4 image-only models, one per target column.


## Part III — Test-set prediction

### 6. Structural checks on the test set

These ask whether the run is *well-formed*, which is a different question from whether it is
*accurate* — and one that has to pass regardless. Each check catches a failure that would
otherwise produce a plausible-looking but wrong submission file rather than an error.

In [8]:
checks = []


def check(name, passed, detail=""):
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))
    return passed


# 1. Every test id has an image file.
missing_imgs = [i for i in TEST_IDS if not (IMAGES_TEST_DIR / f"{i}.jpg").exists()]
check("every test id has an image file", not missing_imgs,
      f"{len(missing_imgs)} missing" if missing_imgs else f"{len(TEST_IDS)} images found")

# 2. Test ids are unique -- a duplicate would make the row order ambiguous.
check("test ids are unique", test_df['id'].is_unique,
      f"{test_df['id'].duplicated().sum()} duplicates")

# 3. Every test image opens and reports a sane size.
bad_imgs, sizes = [], []
for img_id in TEST_IDS[:500]:
    try:
        with Image.open(IMAGES_TEST_DIR / f"{img_id}.jpg") as im:
            sizes.append(im.size)
    except Exception as e:
        bad_imgs.append((img_id, str(e)))
check("test images open cleanly (500-image sample)", not bad_imgs,
      f"most common size {pd.Series(sizes).value_counts().index[0]}" if sizes else "")

# 4. A single batch survives the full transform at the size the models expect.
check("test images pass through eval_transform",
      tuple(probe_imgs.shape[1:]) == (3, IMG_HEIGHT, IMG_WIDTH),
      f"batch tensor {tuple(probe_imgs.shape)}")

# 5. Loader preserves id order -- the assumption every alignment depends on.
check("DataLoader preserves id order", list(probe_ids) == TEST_IDS[:BATCH_SIZE])

# 6. The template has a column for every target we predict.
missing_cols = [c for c in TARGETS if c not in TEST_TEMPLATE_COLUMNS]
check("template has a column for every target", not missing_cols,
      f"missing {missing_cols}" if missing_cols else str(TARGETS))

structural_report = pd.DataFrame(checks)
display(structural_report)

n_failed = (structural_report['result'] == 'FAIL').sum()
if n_failed:
    raise AssertionError(f"{n_failed} structural check(s) failed -- fix before predicting. "
                         "See the table above.")
print("\nAll structural checks passed.")

[PASS] every test id has an image file -- 5829 images found
[PASS] test ids are unique -- 0 duplicates
[PASS] test images open cleanly (500-image sample) -- most common size (60, 80)
[PASS] test images pass through eval_transform -- batch tensor (128, 3, 80, 60)
[PASS] DataLoader preserves id order
[PASS] template has a column for every target -- ['articleType', 'season', 'gender', 'usage']


,check,result,detail
0,every test id has an image file,PASS,5829 images found
1,test ids are unique,PASS,0 duplicates
2,test images open cleanly (500-image sample),PASS,"most common size (60, 80)"
3,test images pass through eval_transform,PASS,"batch tensor (128, 3, 80, 60)"
4,DataLoader preserves id order,PASS,
5,template has a column for every target,PASS,"['articleType', 'season', 'gender', 'usage']"



All structural checks passed.


### 7. Predicting the test set

One pass over the test images; each model's logits are decoded through its own label encoder.
Because Task 1's articleType model was trained on the *grouped* label space and Task 3's
models on the raw `gender` / `usage` columns, `inverse_transform` on each model's own encoder
is the only correct way to get strings back — the encoders are not interchangeable.

In [9]:
test_loader = make_inference_loader(TEST_IDS, IMAGES_TEST_DIR)
print(f"Predicting {len(TEST_IDS)} test images with {len(image_models)} image-only models "
      f"in one pass...")

code_preds = predict_all_image_only(image_models, test_loader)

submission = test_df.copy()
for target in TARGETS:
    model, encoder = image_models[target]
    labels = encoder.inverse_transform(code_preds[target])
    assert len(labels) == len(TEST_IDS), f"{target}: {len(labels)} predictions for {len(TEST_IDS)} ids"
    submission[target] = labels

print("\nPrediction counts per column:")
for target in TARGETS:
    vc = submission[target].value_counts()
    print(f"\n{target} ({vc.size} distinct):")
    print(vc.head(8).to_string())

submission.head()

Predicting 5829 test images with 4 image-only models in one pass...


image-only inference:   0%|          | 0/46 [00:00<?, ?it/s]


Prediction counts per column:

articleType (88 distinct):
articleType
Perfume and Body Mist    411
Lips                     379
Watches                  300
Sarees                   297
Tshirts                  284
Handbags                 270
Kurtas                   229
Flats                    202

season (4 distinct):
season
Summer    2978
Spring    1317
Winter    1035
Fall       499

gender (5 distinct):
gender
Women     3493
Men       2010
Unisex     250
Boys        42
Girls       34

usage (7 distinct):
usage
Casual          4214
Ethnic          1016
Sports           374
Formal           216
Smart Casual       7
Party              1
Travel             1


,id,gender,articleType,season,usage
0,52003,Men,Watches,Fall,Casual
1,52007,Women,Dresses,Summer,Ethnic
2,52017,Men,Fragrance Gift Set,Winter,Casual
3,52021,Women,Sarees,Summer,Casual
4,52023,Men,Innerwear Vests,Summer,Casual


### 8. Final submission checks and save

In [10]:
submission = submission[TEST_TEMPLATE_COLUMNS]   # exact template column order

final_checks = []


def final_check(name, passed, detail=""):
    final_checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


final_check("column names and order match the template",
            list(submission.columns) == TEST_TEMPLATE_COLUMNS, str(list(submission.columns)))
final_check("row count matches the template",
            len(submission) == len(test_df), f"{len(submission)} vs {len(test_df)}")
final_check("id order matches the template", list(submission['id']) == TEST_IDS)
final_check("no empty predictions", submission[TARGETS].notna().all().all(),
            f"{int(submission[TARGETS].isna().sum().sum())} NaN cells")

for target in TARGETS:
    valid = set(image_models[target][1].classes_)
    bad = set(submission[target]) - valid
    final_check(f"every predicted {target} is a valid class label", not bad, str(bad) if bad else "")

# A model that collapsed to a single class would still pass every check above.
for target in TARGETS:
    n_unique = submission[target].nunique()
    top_share = submission[target].value_counts(normalize=True).iloc[0]
    final_check(f"{target} predictions are not degenerate", n_unique > 1 and top_share < 0.95,
                f"{n_unique} distinct values, most common {top_share:.1%}")

display(pd.DataFrame(final_checks))
if (pd.DataFrame(final_checks)['result'] == 'FAIL').any():
    raise AssertionError("Submission checks failed -- do not submit this file.")

SUBMISSION_PATH = OUTPUT_DIR / 'COSC2753_A2_MAI_G8.csv'
submission.to_csv(SUBMISSION_PATH, index=False)

# Reload check: what is on disk is what we think it is.
reloaded = pd.read_csv(SUBMISSION_PATH)
reloaded['id'] = reloaded['id'].astype(str)
assert list(reloaded.columns) == TEST_TEMPLATE_COLUMNS and len(reloaded) == len(test_df)
assert list(reloaded['id']) == TEST_IDS
assert reloaded[TARGETS].notna().all().all()

print(f"\nWrote {len(submission)} image-only predictions to {SUBMISSION_PATH}")
print("\nFirst rows of the submission:")
print(submission.head(10).to_string(index=False))

[PASS] column names and order match the template -- ['id', 'gender', 'articleType', 'season', 'usage']
[PASS] row count matches the template -- 5829 vs 5829
[PASS] id order matches the template
[PASS] no empty predictions -- 0 NaN cells
[PASS] every predicted articleType is a valid class label
[PASS] every predicted season is a valid class label
[PASS] every predicted gender is a valid class label
[PASS] every predicted usage is a valid class label
[PASS] articleType predictions are not degenerate -- 88 distinct values, most common 7.1%
[PASS] season predictions are not degenerate -- 4 distinct values, most common 51.1%
[PASS] gender predictions are not degenerate -- 5 distinct values, most common 59.9%
[PASS] usage predictions are not degenerate -- 7 distinct values, most common 72.3%


,check,result,detail
0,column names and order match the template,PASS,"['id', 'gender', 'articleType', 'season', 'usa..."
1,row count matches the template,PASS,5829 vs 5829
2,id order matches the template,PASS,
3,no empty predictions,PASS,0 NaN cells
4,every predicted articleType is a valid class l...,PASS,
5,every predicted season is a valid class label,PASS,
6,every predicted gender is a valid class label,PASS,
7,every predicted usage is a valid class label,PASS,
8,articleType predictions are not degenerate,PASS,"88 distinct values, most common 7.1%"
9,season predictions are not degenerate,PASS,"4 distinct values, most common 51.1%"



Wrote 5829 image-only predictions to ..\outputs\image_only\COSC2753_A2_MAI_G8.csv

First rows of the submission:
   id gender        articleType season  usage
52003    Men            Watches   Fall Casual
52007  Women            Dresses Summer Ethnic
52017    Men Fragrance Gift Set Winter Casual
52021  Women             Sarees Summer Casual
52023    Men    Innerwear Vests Summer Casual
52026    Men             Shirts Summer Casual
52027    Men        Sweatshirts Summer Formal
52028    Men            Topwear Summer Casual
52029    Men            Jackets Summer Sports
52030    Men            Jackets Summer Casual


## Summary

**What this notebook did:** trained nothing. It took the four models Tasks 1–3 already
selected as their final, deployable, image-only choices — `articleType` (SE-Residual CNN,
Task 1), `season` (Task 2), and `gender`/`usage` (Task 3) — and assembled them into one aligned
inference pipeline over the real, unseen test set.

**What we got:** all required artifacts and checkpoints were present and loaded correctly
(with two architectures identified from their weight shapes rather than saved metadata);
every structural check on the test set passed before prediction; predictions were produced for
all 5,829 test images across all four targets, with plausible, non-degenerate class
distributions; and every final submission check passed, producing
`styles_prediction_filled.csv`.

**What it means:** this file is not just *a* prediction — it is specifically the prediction
that follows from the ultimate judgement reached in Task 1 (and mirrored in Tasks 2–3): an
image-only model, because the real test set never contains the metadata a fused model would
need. Every check in this notebook exists to catch a way that judgement could be silently
undermined at the last step — a wrong architecture guess, a misaligned id, a collapsed
column — rather than to re-argue the modelling decisions themselves.

**Next steps:** if any of Tasks 1–3 retrains or changes its selected checkpoint, this notebook
should simply be re-run — it makes no independent modelling choices, so there is nothing here
to keep in sync manually. A genuine accuracy check on this submission is not possible from
within the team's own data, since the real test labels are never provided; an independent
evaluation would require either held-out labelled data from outside this dataset or the
official grading process itself.